XI coefficient-count update (2026-09-06). Counts use the tested module without decimal rounding, and an empty expansion has zero terms. Outputs were cleared because old counts are not reproducible under the correction. Other historical exploratory code is not a validated research result; see `RESEARCH_VALIDATION.md`.


In [ ]:
import numpy as np
import math
import itertools
import random
import matplotlib as mpl
import matplotlib.pyplot as plt
import qiskit
import qiskit.quantum_info as qi 
# import warnings
# warnings.filterwarnings('ignore')
import time
import networkx as nx
from scipy.linalg import expm
from scipy.optimize import minimize, Bounds
from qiskit import QuantumCircuit, Aer, execute


In [ ]:
from itertools import product
from functools import reduce

PAULIS = {'I': np.eye(2),
          'X': np.array([[0, 1], [1, 0]]),
          'Y': np.array([[0, -1j], [1j, 0]]),
          'Z': np.diag([1, -1])}

def get_pauli_decomp(in_matrix, display_every=5000, PAULIS=PAULIS):
    """Return dictionary with pauli strings and according weight.
    in_matrix - input matrix (should be of size(2^m,2^m), where m is int.
    display_time - if given int number print time spend for computation if False do not print.
    PAULIS - dictionary with Pauli basis.

    """
    m = int(np.log2(in_matrix.shape[0]))

    pauli_weights = {}

    k = 1
    K = len(list(product(PAULIS.keys(), repeat=m)))

    start_ = time.time()
    for u in product(PAULIS.keys(), repeat=m):
        pauli_str_name = ''.join(u)
        pauli_str_matrix = reduce(np.kron, [PAULIS[s] for s in u])
        inner_product = np.trace(in_matrix @ pauli_str_matrix) / 2**m
        if not np.isclose(inner_product, 0):
            pauli_weights[pauli_str_name] = inner_product
        if display_every and k%display_every == 0:
            print('\t {} qubits || {}/{} || {:.3f} s passed'.format(m,k,K, time.time()-start_))
        k+=1

    return pauli_weights

# produce all binary strings of length n with k 1s. If k is None then all possible binary strings of length n produced
def get_binary_strings(n, k=None) -> list[list]:
    '''
    produce all binary strings of length n with k 1s
    returns list with binary lists 
    '''
    final = []
    def kbits(r):
        result = []
        for bits in itertools.combinations(range(n), r):
            s = [0] * n
            for bit in bits:
                s[bit] = 1
            result.append(s)   
        return result

    if k != None:
        return kbits(k)
    
    for i in range(n + 1):
        final = final + kbits(i)
        
    return final

def count_solutions_XI(nqubits, Operator, tol=1e-10, if_print=False):
    from QAOA_locality import count_solutions_XI as count_exact
    if np.shape(Operator) != (2**nqubits, 2**nqubits):
        raise ValueError("Operator size does not match nqubits.")
    result = count_exact(Operator, tol)
    if if_print:
        print("XI count, maximum locality, average locality:", result[3], result[1], result[2])
    return result



def get_circuit_operators( lst_x=None, lst_z=None):
    '''
    note the order! (x,z)
    '''
    # returns for example X @ I @ I @ X @ I where @ is tensor product
    return  qi.Pauli((lst_z,lst_x)).to_matrix()


def generate_QAOA_operator(H1,H2, number_of_layers = 1 , beta_angle = [], gamma_angle = []):
    """_summary_

    Args:
        H (_type_): _description_
        number_of_layers (int, optional): _description_. Defaults to 1.
        beta_angle (list, optional): _description_. Defaults to [].
        gamma_angle (list, optional): _description_. Defaults to [].

    Returns:
        _type_: _description_
    """    '''
    '''
    # define angles for mixers 
    if len(beta_angle)==0:
        beta_angle = np.random.rand(number_of_layers) * np.pi/3


    if len(gamma_angle)==0:
        gamma_angle = np.random.rand(number_of_layers) * np.pi/6

    ans = H1

    nqubits = int(np.log2(len(H1)))

   
    mixer = 0
    x_string = np.zeros(nqubits)

    for i in range(0,nqubits):
        x_string[i] = 1
        mixer = mixer + get_circuit_operators(x_string, np.zeros(nqubits))
        x_string[i] = 0
    for p in range(number_of_layers):
        # Making unitary for Hamiltonian exp
        exp_H = expm(1j*H2*beta_angle[p])
        # Making mixer X
        exp_X = expm(1j*mixer*gamma_angle[p])

        ans = exp_H @ exp_X @ ans @ exp_X.T.conjugate() @ exp_H.T.conjugate()

    return ans
   
def make_H_maxCUT(n, adj_mat):
    H = np.zeros((2**n,2**n),dtype=complex)
    zeros = np.zeros(n)

    for i in range(n):
        for j in range(i+1,n):
            if adj_mat[i][j]!=0:
                buf = np.zeros(n)
                buf[i] = buf[j] = 1
                H += adj_mat[i][j]*get_circuit_operators(zeros, buf)
    return H

In [ ]:
H = make_H_maxCUT(2, [[0,1],[1,0]])
H

In [ ]:
get_pauli_decomp(generate_QAOA_operator(H,H))

In [ ]:
N = 4
# H = np.diag(np.random.rand(2**N))
# generate_QAOA_operator(H)

In [ ]:
def mixed_Hamiltonian(N,*localities):
    rng = np.random.default_rng()
    H = 0
    for l in localities:
        Zs = get_binary_strings(N,l)
        pickedZ = rng.choice(Zs, min(5,len(Zs)), replace=False)
        for i in range(0,len(pickedZ)):
            H = H + get_circuit_operators(np.zeros(N),pickedZ[i])
    return H

In [ ]:
rng = np.random.default_rng()
Zs = get_binary_strings(N,2)
# pickedZ = np.random.choice(Zs,size=number_of_terms,replace=False)
pickedZ = rng.choice(Zs, 4, replace=False)

# Making Hamiltonian
H1 = 0
for i in range(0,len(pickedZ)):
    H1 = H1 + get_circuit_operators(np.zeros(N),pickedZ[i])

buf = np.copy(H1)

In [ ]:
Zs = get_binary_strings(N,3)
# pickedZ = np.random.choice(Zs,size=number_of_terms,replace=False)
pickedZ = rng.choice(Zs, 4, replace=False)

# Making Hamiltonian
H2 = buf
for i in range(0,len(pickedZ)):
    H2 = H2 + get_circuit_operators(np.zeros(N),pickedZ[i])

In [ ]:
count_solutions_XI(N,generate_QAOA_operator(H2,H2,2))

In [ ]:
get_pauli_decomp(generate_QAOA_operator(H1,H1))

In [ ]:
mixed_Hamiltonian(2,1,2)

In [ ]:
N = 5
H = mixed_Hamiltonian(N,1,2,3)
ans , *_= count_solutions_XI(N,generate_QAOA_operator(H,H,1))
# print( np.ones(N) in ans)
ans

1   -- ,
2   --,
3   --,

1+2  2,
1+3  2,
2+3  2,

In [ ]:
N = 5
H = mixed_Hamiltonian(N,2)

In [ ]:
beta_angle = np.random.rand(10) * np.pi/3

gamma_angle = np.random.rand(10) * np.pi/6

ans = np.copy(H)

nqubits = int(np.log2(len(H)))

mixer = 0
x_string = np.zeros(nqubits)

for i in range(0,nqubits):
    x_string[i] = 1
    mixer = mixer + get_circuit_operators(x_string, np.zeros(nqubits))
    x_string[i] = 0
p=0

In [ ]:
# Making unitary for Hamiltonian exp
exp_H = expm(1j*H*beta_angle[p])
# Making mixer X
exp_X = expm(1j*mixer*gamma_angle[p])

ans = exp_H @ exp_X @ ans @ exp_X.T.conjugate() @ exp_H.T.conjugate()

p+=1


In [ ]:
xs = get_circuit_operators(np.ones(nqubits), np.zeros(nqubits))

In [ ]:
exp_H @ xs == xs @ exp_H

In [ ]:
beta = np.random.rand(1) * np.pi/3
beta_p = np.pi/2 - beta 
beta_p

In [ ]:
np.cos(beta), np.sin(beta_p), np.sin(beta), np.cos(beta_p)

In [ ]:
exp_Hx0 = expm(mixer*beta)
exp_Hxp = expm(mixer*beta_p)

In [ ]:
exp_Hx0@xs 

In [ ]:
exp_Hxp/10

In [ ]:
np.trace(exp_Hx0@xs)/2**N

In [ ]:
exp_Hxp

In [ ]:
def plus_state(n):
    return 1/np.sqrt(2**n) * np.ones(2**n)

In [ ]:
def output_state(theta, H, layer):
    n = int(np.log2(len(H)))
    circ = generate_QAOA_operator(H, number_of_layers = layer , beta_angle = theta[:len(theta)//2], gamma_angle = theta[len(theta)//2:])
    return np.dot(plus_state(n).T,circ)


def energy(theta, H, layer):
    state = output_state(theta, H, layer)
    n = int(np.log2(len(H)))
    return np.real(np.dot(state.conjugate().T, np.dot(H, state)))


def get_expectation(H, layer):

    def execute_circ(theta):
        return energy(theta, H, layer)

    return execute_circ


In [ ]:
def theta_for_ground_state(H,layer=1, repeat=10, maxiter=100):
    bds = Bounds(0, 2 * np.pi)
    expectation = get_expectation(H,layer)
    for i in range(repeat):
        initial_params = np.random.uniform(0, np.pi/4, layer*2)  # initial guess of parameters
        res = minimize(expectation,
                       initial_params, method='L-BFGS-B', jac='3-point', bounds=bds, options={'maxiter': maxiter})
        if i == 0:
            E_min = res.fun
            params = res.x
        if res.fun < E_min:
            E_min = res.fun
            params = res.x
        # print(E_min)
    return params, E_min